# Dow Structure Predictive Study v0.3

固定済みの価格とv0.2成果物を用い、モメンタム・ボラティリティ・市場内順位に確定スイング4状態を追加した予測情報の増分を評価します。主判断は上位10%研究群の63日大幅下落予測におけるModel A対Bです。

**新しい価格取得、既存エンジン・v0.1/v0.2の変更、本番スクリーニング・発注・TTL・配分変更は行いません。** 時間順評価と再実行一致を確認しても、既に探索した期間が真正な未使用アウトオブサンプルになるわけではありません。

実際の固定Driveデータ・Colab動作は合成データテストと別です。詳細は `docs/dow_structure_predictive_study_v03.md` を参照してください。


## 1. 固定入力を保存したGoogle Driveを接続

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. 固定入力・起点SHA・パラメータを確認

主分析と感度分析は別段階です。まず `RUN_SENSITIVITIES=False` のまま主分析と監査を完了させます。正常終了後、必要ならTrueにしてセル6だけ実行します。感度結果は別の新規分析フォルダへ保存し、完成済み主分析を変更しません。

固定仕様は63観測区間の最大ドローダウン≤−20%、主系列株価、定点SPAN5・位相0、Model A/Bの同一学習・評価集合です。予測改善や副目的変数を見て主目的を変更しないでください。


In [ ]:
REPO_URL = 'https://github.com/avalon19920629-create/Alpha-Engine-backtest.git'
CODE_REF = 'codex/dow-structure-predictive-v0-3'
EXPECTED_BASE_SHA = '1b2ecb4267d176b327d3995ed0bbc3d7a2b273f6'
SNAPSHOT_DIR = '/content/drive/MyDrive/alpha_engine/research_runs/20261104T124443Z_residual100_n6_9b9d99b92dcf'
V02_DIR = '/content/drive/MyDrive/alpha_engine/analysis/dow_structure_v02/20261111T010459Z_34a3f71bec76'
OUTPUT_ROOT = '/content/drive/MyDrive/alpha_engine/analysis/dow_predictive_v03'
BOOTSTRAP_SAMPLES = 500
SEED = 20261111
RUN_SENSITIVITIES = False
SENSITIVITY_OUTPUT_ROOT = None  # None: 主分析と並ぶ新規フォルダへ保存


## 3. 入力存在確認・専用ブランチのコミット固定・依存関係導入

コードは新しい `/content` チェックアウトへ取得します。既存作業へのpull/resetは行いません。起点の子孫であることと、起点の既存追跡ファイルが変更・削除・置換されていないことを確認します。GitとPythonパッケージの取得は市場データ取得とは別です。入力不足を新しい研究実行で補いません。


In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

snapshot = Path(SNAPSHOT_DIR).expanduser().resolve()
v02 = Path(V02_DIR).expanduser().resolve()
output = Path(OUTPUT_ROOT).expanduser().resolve()
for input_dir in (snapshot, v02):
    if not input_dir.is_dir():
        raise FileNotFoundError(f'固定入力が見つかりません: {input_dir}。新規価格取得では代替しません。')
    if output == input_dir or input_dir in output.parents:
        raise ValueError('出力先は価格・v0.2入力フォルダの外側にしてください。')
required_snapshot = (
    'price_data/price_cache.parquet', 'price_data/benchmark_prices.parquet',
    'price_data/price_cache_metadata.json', 'research_run_metadata.json',
)
required_v02 = (
    'daily_structure_features.parquet', 'landmark_observations.csv',
    'structure_outcomes.csv', 'structure_outcome_summary.csv',
    'extreme_transitions.csv', 'study_metadata.json', 'file_manifest.json',
)
for folder, names in ((snapshot, required_snapshot), (v02, required_v02)):
    for name in names:
        if not (folder / name).is_file():
            raise FileNotFoundError(f'固定入力が不足しています: {folder / name}')

REPO_DIR = Path(tempfile.mkdtemp(prefix='dow-predictive-study-', dir='/content'))
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)

def resolve_code_ref(ref):
    for candidate in (ref, f'refs/remotes/origin/{ref}'):
        result = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'rev-parse', '--verify', '--end-of-options',
             f'{candidate}^{{commit}}'], text=True, capture_output=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'clone済み参照を解決できません: {ref}')

CODE_SHA = resolve_code_ref(CODE_REF)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor', EXPECTED_BASE_SHA, CODE_SHA],
    check=True,
)
existing_changes = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'diff', '--name-status', '--diff-filter=CDMRTUXB',
     EXPECTED_BASE_SHA, CODE_SHA, '--'], text=True, capture_output=True, check=True,
)
if existing_changes.stdout.strip():
    raise RuntimeError('既存ファイルの変更を検出しました:\n' + existing_changes.stdout)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_SHA], check=True)
print('Pinned predictive code:', CODE_REF, CODE_SHA)
print('Unchanged existing-code base:', EXPECTED_BASE_SHA)
print('Fixed snapshot:', snapshot)
print('Fixed v0.2 analysis:', v02)
print('New analysis output root:', output)
assert (REPO_DIR / 'dow_structure_predictive_study.py').is_file()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-dow-predictive.txt')],
    check=True,
)


## 4. 主仕様：特徴量・目的変数・逐次予測・独立再実行監査

全ての分類可能な定点を使い、v0.2でマッチした標本だけへ絞りません。主な進行段階と予測日ごとの学習・予測件数は実行ログで確認できます。未来のラベルが未観測でも、その日までの特徴量で予測可能な行は予測を保存します。

主分析ではウォークフォワードを独立に再実行して予測一致を自動照合します。計算時間を短縮するための検査省略オプションは設けません。再実行一致は数値再現性の監査で、未使用期間の予測優位性の証明ではありません。


In [ ]:
import json

MAIN_RUN_DIR = None
MAIN_RECEIPT = Path(tempfile.mkdtemp(
    prefix='dow-predictive-main-receipt-', dir=str(REPO_DIR.parent),
)) / 'result.json'
run_config = {
    'snapshot_dir': str(snapshot), 'v02_dir': str(v02), 'output_root': str(output),
    'bootstrap_samples': BOOTSTRAP_SAMPLES, 'seed': SEED,
}
child_code = r"""
import json
import sys
from pathlib import Path
from dow_structure_predictive_study import run_study

receipt = Path(sys.argv[2])
try:
    run_dir = run_study(**json.loads(sys.argv[1]))
except Exception as exc:
    receipt.write_text(json.dumps({
        'call_status': 'failed', 'run_dir': str(getattr(exc, 'run_dir', '') or ''),
        'error': str(exc),
    }, ensure_ascii=False), encoding='utf-8')
    raise
else:
    receipt.write_text(json.dumps({'call_status': 'returned', 'run_dir': str(run_dir)}), encoding='utf-8')
"""
result = subprocess.run(
    [sys.executable, '-c', child_code, json.dumps(run_config), str(MAIN_RECEIPT)],
    cwd=REPO_DIR,
)
main_receipt = json.loads(MAIN_RECEIPT.read_text()) if MAIN_RECEIPT.exists() else {
    'call_status': 'failed', 'run_dir': '', 'error': '今回の通知がありません。上の実行エラーを確認してください。',
}
MAIN_RUN_DIR = Path(main_receipt['run_dir']) if main_receipt.get('run_dir') else None
print('Main process exit code:', result.returncode)
print('Main receipt:', main_receipt)
if result.returncode:
    print('主分析は失敗しました。部分成果物をcompletedとして扱わないでください。')


## 5. Model A対Bの主比較と評価可能件数

主判断は上位10%群の `deep_drawdown_63` です。全候補・米国・日本・3M符号別も確認します。標本不足は不支持や研究凍結と同義ではありません。Model 0/A/Bで同じ観測を比較しているか、学習不足・右側打切り・価格欠損・既知異常・計算失敗を別々に読みます。

副目的変数が良好でも主目的の不支持を覆してGate Aへ昇格させません。主分析の予測は評価指標を計算する前に保存します。

目的変数・モデルの実計算失敗は `status=partial` とGate評価不能になります。他の正常な診断値は保存します。訓練不足・リスク対象外・打切りは通常の状態として区別し、`target_status_counts.csv` と品質記録の `target_error` を確認してください。


In [ ]:
import pandas as pd
from IPython.display import JSON, Markdown, display

if MAIN_RUN_DIR is None:
    raise RuntimeError('今回の主分析保存先が不明です。前回の結果ではなく実行セルのエラーを確認してください。')
metadata_path = MAIN_RUN_DIR / 'study_metadata.json'
if metadata_path.is_file():
    main_meta = json.loads(metadata_path.read_text(encoding='utf-8'))
    display(JSON(main_meta))
    if main_meta.get('status') != 'completed' or main_meta.get('validation_status') != 'passed':
        print('主分析は未完了または検査失敗です。完成済み研究結果として扱わないでください。')
else:
    print('未作成:', metadata_path)
primary_risk_path = MAIN_RUN_DIR / 'risk_metrics.csv'
if primary_risk_path.is_file():
    try:
        primary_risk = pd.read_csv(primary_risk_path)
    except pd.errors.EmptyDataError:
        primary_risk = pd.DataFrame()
    required_columns = {'target', 'cohort', 'series_kind', 'momentum_3m_sign'}
    if required_columns.issubset(primary_risk.columns):
        primary_risk = primary_risk.loc[
            primary_risk.target.eq('deep_drawdown_63')
            & primary_risk.cohort.eq('raw_top10')
            & primary_risk.series_kind.eq('price')
            & primary_risk.momentum_3m_sign.eq('all')
        ]
        display(Markdown('**主比較：上位10%・株価構造・63日大幅下落（ALL/US/JP、Model 0/A/B）**'))
        display(primary_risk)
priority = (
    'risk_metrics.csv', 'subgroup_metrics.csv', 'uncertainty.csv',
    'momentum_persistence_metrics.csv', 'calibration.csv',
    'time_half_metrics.csv', 'top10_retention.csv', 'target_status_counts.csv',
    'correction_log.csv', 'walkforward_schedule.csv', 'training_audit.csv',
    'quality_warnings.csv',
)
paths = [MAIN_RUN_DIR / name for name in priority if (MAIN_RUN_DIR / name).is_file()]
paths += [path for path in sorted(MAIN_RUN_DIR.glob('*.csv')) if path not in paths]
for path in paths:
    display(Markdown(f'**{path.name}（先頭60行）**'))
    try:
        display(pd.read_csv(path, nrows=60))
    except pd.errors.EmptyDataError:
        print('記録行なし（ファイルは空です）。')


## 6. 副目的変数と、任意の別段階感度分析

主分析には順位変化・3M負転換・相対リターンの副目的も含まれます。SPAN・位相・系列・補正の感度分析は、主分析の正常完了とマニフェスト照合後に別の新規フォルダへ保存します。

`RUN_SENSITIVITIES=True` にした場合だけ実行します。主分析フォルダやGate判定を書き換えず、感度分析の最良結果を主結果へ昇格させません。

主分析側の `sensitivity_analysis.csv` は未実行案内のまま保存します。副段階は主分析のseed・反復数を継承し、七設定の結果を新規兄弟フォルダへ保存します。設定別の独立再実行は省くため各設定Gateは評価不能で、主分析承認へ置き換えません。


In [ ]:
SENSITIVITY_RUN_DIR = None
if not RUN_SENSITIVITIES:
    print('感度分析は未実行です。主分析の監査完了後、必要なら設定セルでTrueにしてこのセルを実行してください。')
else:
    if MAIN_RUN_DIR is None:
        raise RuntimeError('正常完了した今回の主分析が必要です。')
    metadata = json.loads((MAIN_RUN_DIR / 'study_metadata.json').read_text(encoding='utf-8'))
    if metadata.get('status') != 'completed' or metadata.get('validation_status') != 'passed':
        raise RuntimeError('主分析がcompletedかつvalidation passedである必要があります。')
    sensitivity_receipt_path = Path(tempfile.mkdtemp(
        prefix='dow-predictive-sensitivity-receipt-', dir=str(REPO_DIR.parent),
    )) / 'result.json'
    sensitivity_config = {'main_run_dir': str(MAIN_RUN_DIR), 'output_root': SENSITIVITY_OUTPUT_ROOT}
    sensitivity_child = r"""
import json
import sys
from pathlib import Path
from dow_structure_predictive_study import run_sensitivities

receipt = Path(sys.argv[2])
try:
    run_dir = run_sensitivities(**json.loads(sys.argv[1]))
except Exception as exc:
    receipt.write_text(json.dumps({
        'call_status': 'failed', 'run_dir': str(getattr(exc, 'run_dir', '') or ''),
        'error': str(exc),
    }, ensure_ascii=False), encoding='utf-8')
    raise
else:
    receipt.write_text(json.dumps({'call_status': 'returned', 'run_dir': str(run_dir)}), encoding='utf-8')
"""
    sensitivity_result = subprocess.run(
        [sys.executable, '-c', sensitivity_child, json.dumps(sensitivity_config), str(sensitivity_receipt_path)],
        cwd=REPO_DIR,
    )
    sensitivity_receipt = json.loads(sensitivity_receipt_path.read_text()) if sensitivity_receipt_path.exists() else {
        'call_status': 'failed', 'run_dir': '', 'error': '今回の副分析通知がありません。実行ログを確認してください。',
    }
    SENSITIVITY_RUN_DIR = Path(sensitivity_receipt['run_dir']) if sensitivity_receipt.get('run_dir') else None
    print('Sensitivity process exit code:', sensitivity_result.returncode)
    print('Sensitivity receipt:', sensitivity_receipt)
    if SENSITIVITY_RUN_DIR is not None:
        sensitivity_meta = SENSITIVITY_RUN_DIR / 'study_metadata.json'
        if sensitivity_meta.is_file():
            sensitivity_metadata = json.loads(sensitivity_meta.read_text(encoding='utf-8'))
            display(JSON(sensitivity_metadata))
            if sensitivity_metadata.get('status') != 'completed' or sensitivity_metadata.get('validation_status') != 'passed':
                print('副分析は未完了・部分失敗・検査失敗です。完成済み結果として扱わないでください。')
        table = SENSITIVITY_RUN_DIR / 'sensitivity_analysis.csv'
        if table.is_file():
            try:
                display(pd.read_csv(table))
            except pd.errors.EmptyDataError:
                print('感度集計の記録行がありません。実行状態を確認してください。')


## 7. Gate I・監査レポート・図・保存先

Gate Aは本番導入許可ではありません。主評価の有効標本不足・8暦月未満は `evaluation_unavailable` としてC（研究凍結）から区別します。片市場だけが支持する場合は市場限定の候補です。銘柄クラスタ区間は補助診断、主不確実性は市場月ブロックで、較正済み二方向クラスタ検定とは呼びません。

予測の独立再実行一致は入力・処理・数値の監査であり、既に探索した期間を未観測OOSに変えません。原本・過去研究を保持し、今後の工学的検証や本番採否は別工程です。

総合Aにはプール条件と少なくとも一市場の個別Aが必要です。片市場だけがAなら適用市場を限定し、個別Aだけで非Aのプール主判定を昇格させません。


In [ ]:
from IPython.display import SVG

if MAIN_RUN_DIR is None:
    raise RuntimeError('今回の主分析フォルダがありません。')
for name in ('gate1_assessment.json', 'file_manifest.json'):
    path = MAIN_RUN_DIR / name
    if path.is_file():
        display(Markdown(f'**{name}**'))
        display(JSON(json.loads(path.read_text(encoding='utf-8'))))
for name in ('gate1_assessment.md', 'study_report.md'):
    path = MAIN_RUN_DIR / name
    if path.is_file():
        display(Markdown(path.read_text(encoding='utf-8')))
chart_paths = sorted((MAIN_RUN_DIR / 'charts').glob('*.svg'))
print('Main SVG charts:', len(chart_paths))
for path in chart_paths:
    display(Markdown(f'**{path.name}**'))
    display(SVG(filename=str(path)))
print('Fixed snapshot:', snapshot)
print('Fixed v0.2 inputs:', v02)
print('Main analysis:', MAIN_RUN_DIR)
print('Sensitivity analysis:', globals().get('SENSITIVITY_RUN_DIR') or '(not run or unavailable)')
print('Pinned v0.3 code SHA:', CODE_SHA)
